In [ ]:
# Import required libraries

from google.colab import drive

import os
import sys
import json
import time
import hashlib
import platform
import gc
import re

from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import transformers
import pyarrow
import psutil
import matplotlib.pyplot as plt

from IPython.display import display


# Mount Google Drive

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir()


# Define project directories

CONFIG_DIR = PROJECT_ROOT / '00_config'
DATA_DIR = PROJECT_ROOT / '01_data'
PROCESSED_DIR = DATA_DIR / 'processed'
SPLITS_DIR = DATA_DIR / 'splits'
FEATURES_DIR = DATA_DIR / 'features'
TOKENIZED_DIR = DATA_DIR / 'tokenized'
SRC_DIR = PROJECT_ROOT / '03_src'
MODELS_DIR = PROJECT_ROOT / '04_models'

NOTEBOOK_DIR = (
    PROJECT_ROOT / '02_notebooks/07_final_evaluation'
)

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'

for directory in [
    NOTEBOOK_DIR,
    RESULTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    PREDICTIONS_DIR
]:

    directory.mkdir(parents=True, exist_ok=True)


# Define previous notebook artifact paths

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

OPTIMIZATION_RESULTS_DIR = (
    PROJECT_ROOT
    / '02_notebooks/06_model_optimization/results'
)

FROZEN_MANIFEST_PATH = (
    OPTIMIZATION_RESULTS_DIR
    / 'configs/frozen_manifest.json'
)

OPTIMIZATION_COMPLETION_PATH = (
    OPTIMIZATION_RESULTS_DIR
    / 'logs/optimization_completion_summary.json'
)


# Define reusable file utilities

def read_json(path):

    with open(path, 'r') as file:
        return json.load(file)


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b''
        ):
            digest.update(chunk)

    return digest.hexdigest()


def save_json_atomic(path, content):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(
        path.suffix + '.tmp'
    )

    with open(temporary_path, 'w') as file:

        json.dump(
            content,
            file,
            indent=4,
            allow_nan=False
        )

    assert read_json(temporary_path) == content

    temporary_path.replace(path)


def save_csv_atomic(path, dataframe):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(
        path.suffix + '.tmp'
    )

    dataframe.to_csv(temporary_path, index=False)

    temporary_path.replace(path)


def verify_frozen_file(record):

    path = (
        PROJECT_ROOT / record['path']
    ).resolve()

    assert path.is_relative_to(PROJECT_ROOT.resolve())
    assert path.is_file(), f"Missing file: {path}"

    assert path.stat().st_size == record['size_bytes']

    assert calculate_sha256(path) == record['sha256'], (
        f"Fingerprint mismatch: {path}"
    )

    return path


# Verify the optimization completion gate

optimization_completion = read_json(
    OPTIMIZATION_COMPLETION_PATH
)

assert optimization_completion['status'] == 'Completed'
assert optimization_completion['final_project_model_frozen']
assert not optimization_completion['official_test_dataset_loaded']

EXPECTED_FROZEN_HASH = (
    '229adb5d6166a626f6b7f7fddecf38b4de'
    '58d21e56ec4367465ad29174ef2fd1'
)

frozen_manifest_sha256 = calculate_sha256(
    FROZEN_MANIFEST_PATH
)

assert frozen_manifest_sha256 == EXPECTED_FROZEN_HASH

assert frozen_manifest_sha256 == (
    optimization_completion['frozen_manifest_sha256']
)

frozen_manifest = read_json(FROZEN_MANIFEST_PATH)

assert frozen_manifest['selection_partition'] == 'validation'
assert frozen_manifest['thresholds_frozen']
assert frozen_manifest['model_parameters_frozen']
assert frozen_manifest['final_project_model_frozen']
assert frozen_manifest['test_evaluation_permitted']

assert not frozen_manifest['official_test_used_for_selection']
assert not frozen_manifest['post_freeze_refitting_permitted']

print("Frozen evaluation protocol verified.")


# Restore frozen model information

LABELS = frozen_manifest['labels']
NUM_LABELS = len(LABELS)
SEED = 42

PRIMARY_RUN_ID = frozen_manifest['primary_selected_run_id']
EVALUATION_ROSTER = frozen_manifest['evaluation_roster']
SELECTED_MODELS = frozen_manifest['selected_models']

FROZEN_MODELS = {
    entry['run_id']: entry
    for entry in frozen_manifest['models']
}

assert LABELS == [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

assert NUM_LABELS == 6
assert len(EVALUATION_ROSTER) == 19
assert len(FROZEN_MODELS) == 19

assert set(EVALUATION_ROSTER) == set(FROZEN_MODELS)

assert PRIMARY_RUN_ID == 'S02_ROBERTA_FOCAL_FULL_S42'

CLASSICAL_RUN_IDS = [
    rid for rid in EVALUATION_ROSTER
    if FROZEN_MODELS[rid]['kind'] == 'classical'
]

TRANSFORMER_RUN_IDS = [
    rid for rid in EVALUATION_ROSTER
    if FROZEN_MODELS[rid]['kind'] == 'transformer'
]

assert len(CLASSICAL_RUN_IDS) == 12
assert len(TRANSFORMER_RUN_IDS) == 7

for rid, entry in FROZEN_MODELS.items():

    assert entry['labels'] == LABELS
    assert entry['comparison_operator'] == '>='

    assert set(entry['optimized_thresholds']) == set(LABELS)

    assert np.isfinite([
        entry['optimized_thresholds'][label]
        for label in LABELS
    ]).all()

print("Frozen configurations:", len(FROZEN_MODELS))
print("Classical configurations:", len(CLASSICAL_RUN_IDS))
print("Transformer configurations:", len(TRANSFORMER_RUN_IDS))
print("Frozen thresholds:", len(FROZEN_MODELS) * NUM_LABELS)
print("Primary selected model:", PRIMARY_RUN_ID)


# Verify registry and methodology records

verify_frozen_file(frozen_manifest['registry'])

assert calculate_sha256(DECISION_LOG_PATH) == (
    frozen_manifest['decision_log_sha256']
)

registry = pd.read_csv(REGISTRY_PATH)
decisions = pd.read_csv(DECISION_LOG_PATH)

assert len(registry) == 19
assert len(decisions) == 74
assert registry['run_id'].is_unique
assert registry['status'].eq('Completed').all()

assert set(registry['run_id']) == set(EVALUATION_ROSTER)

print("Experiment registry verified.")
print("Methodological decisions verified.")


# Restore shared evaluation functions

verify_frozen_file(frozen_manifest['evaluation_module'])

sys.path.insert(0, str(SRC_DIR))

import evaluation_utils

from evaluation_utils import (
    evaluate_multilabel,
    apply_thresholds
)

assert Path(evaluation_utils.__file__).resolve() == (
    SRC_DIR / 'evaluation_utils.py'
).resolve()

print("Shared evaluation module verified.")


# Load the official scored test dataset

test_specification = frozen_manifest['official_test']

TEST_PATH = (
    PROJECT_ROOT / test_specification['dataset_path']
)

EXPECTED_TEST_SAMPLES = test_specification['expected_rows']
EXPECTED_TEST_HASH = test_specification['expected_sha256']

assert EXPECTED_TEST_SAMPLES == 63978
assert TEST_PATH.is_file()

assert calculate_sha256(TEST_PATH) == EXPECTED_TEST_HASH

test_df = pd.read_parquet(
    TEST_PATH,
    columns=['id', 'comment_text', *LABELS]
)

assert len(test_df) == EXPECTED_TEST_SAMPLES
assert test_df['id'].notna().all()
assert test_df['id'].is_unique

assert test_df['comment_text'].notna().all()

assert test_df['comment_text'].map(
    lambda value: isinstance(value, str)
).all()

assert np.isin(
    test_df[LABELS].to_numpy(),
    [0, 1]
).all()

y_test = test_df[LABELS].to_numpy(dtype=np.int8)
test_ids = test_df['id'].astype(str).to_numpy()

assert y_test.shape == (EXPECTED_TEST_SAMPLES, NUM_LABELS)
assert (y_test.sum(axis=0) > 0).all()

assert (
    y_test.sum(axis=0) < EXPECTED_TEST_SAMPLES
).all()

print("\nOfficial scored test dataset loaded.")
print("Test comments:", len(test_df))
print("Dataset fingerprint: Verified")
print("Unique IDs:", len(np.unique(test_ids)))
print("Unavailable labels: 0")


# Verify official test ID alignment

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

test_manifest_rows = dataset_manifest.loc[
    dataset_manifest['Partition'].eq('test_official')
]

assert len(test_manifest_rows) == 1

test_manifest_row = test_manifest_rows.iloc[0]

assert int(test_manifest_row['Rows']) == EXPECTED_TEST_SAMPLES
assert test_manifest_row['SHA256'] == EXPECTED_TEST_HASH

TEST_ID_MANIFEST_PATH = (
    PROJECT_ROOT / test_manifest_row['Manifest Path']
)

test_id_manifest = pd.read_csv(
    TEST_ID_MANIFEST_PATH,
    dtype={'id': str}
)

np.testing.assert_array_equal(
    test_id_manifest['id'].to_numpy(),
    test_ids
)

if 'row_index' in test_id_manifest.columns:

    np.testing.assert_array_equal(
        test_id_manifest['row_index'].to_numpy(),
        np.arange(EXPECTED_TEST_SAMPLES)
    )

print("Official test ID alignment verified.")


# Construct duplicate-aware groups for bootstrap analysis

normalized_test_text = test_df['comment_text'].map(
    lambda text: re.sub(
        r'\s+',
        ' ',
        text.lower()
    ).strip()
)

test_group_ids, unique_test_groups = pd.factorize(
    normalized_test_text,
    sort=False
)

TEST_GROUP_COUNT = len(unique_test_groups)

assert len(test_group_ids) == EXPECTED_TEST_SAMPLES
assert (test_group_ids >= 0).all()

test_group_manifest = pd.DataFrame({
    'id': test_ids,
    'row_index': np.arange(EXPECTED_TEST_SAMPLES),
    'bootstrap_group_id': test_group_ids
})

save_csv_atomic(
    TABLES_DIR / 'test_bootstrap_groups.csv',
    test_group_manifest
)

print("Normalized test groups:", TEST_GROUP_COUNT)

del normalized_test_text, unique_test_groups


# Record official test label support

test_support_df = pd.DataFrame({
    'Label': LABELS,
    'Positive Support': y_test.sum(axis=0),
    'Negative Support': (
        EXPECTED_TEST_SAMPLES - y_test.sum(axis=0)
    ),
    'Prevalence (%)': y_test.mean(axis=0) * 100
})

print("\nOfficial test label support")

display(test_support_df.round(4))

save_csv_atomic(
    TABLES_DIR / 'official_test_label_support.csv',
    test_support_df
)


# Prepare the frozen evaluation roster

roster_records = []

for rank, rid in enumerate(EVALUATION_ROSTER, start=1):

    entry = FROZEN_MODELS[rid]

    roster_records.append({
        'Validation Rank': rank,
        'Run ID': rid,
        'Model': entry['model'],
        'Kind': entry['kind'],
        'Training Samples': entry['training_samples'],
        'Score Type': entry['score_type'],
        'Default Threshold': entry['default_threshold'],
        'Validation Macro AP': entry['validation_macro_ap'],
        'Validation Optimized Macro F1': (
            entry['validation_optimized_macro_f1']
        ),
        'Primary Selected Model': rid == PRIMARY_RUN_ID
    })

roster_df = pd.DataFrame(roster_records)

save_csv_atomic(
    TABLES_DIR / 'frozen_evaluation_roster.csv',
    roster_df
)

print("\nFrozen evaluation roster")

display(roster_df.round(6))


# Record runtime resources

CUDA_AVAILABLE = torch.cuda.is_available()

DEVICE = torch.device(
    'cuda' if CUDA_AVAILABLE else 'cpu'
)

environment_info = {
    'python_version': platform.python_version(),
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'scipy_version': scipy.__version__,
    'sklearn_version': sklearn.__version__,
    'torch_version': str(torch.__version__),
    'transformers_version': transformers.__version__,
    'pyarrow_version': pyarrow.__version__,
    'cuda_available': CUDA_AVAILABLE,
    'gpu_name': (
        torch.cuda.get_device_name(0)
        if CUDA_AVAILABLE else None
    ),
    'available_ram_gib': (
        psutil.virtual_memory().available / 1024**3
    ),
    'local_disk_free_gib': (
        psutil.disk_usage('/content').free / 1024**3
    )
}

print("\nRuntime resources")

for key, value in environment_info.items():
    print(f"{key}: {value}")

save_json_atomic(
    LOGS_DIR / 'evaluation_environment.json',
    environment_info
)


# Save evaluation setup

evaluation_setup = {
    'notebook': '07_final_evaluation',
    'frozen_manifest_path': str(
        FROZEN_MANIFEST_PATH.relative_to(PROJECT_ROOT)
    ),
    'frozen_manifest_sha256': frozen_manifest_sha256,
    'primary_selected_run_id': PRIMARY_RUN_ID,
    'evaluation_roster': EVALUATION_ROSTER,
    'classical_run_ids': CLASSICAL_RUN_IDS,
    'transformer_run_ids': TRANSFORMER_RUN_IDS,
    'labels': LABELS,
    'seed': SEED,
    'test_dataset_sha256': EXPECTED_TEST_HASH,
    'test_samples': EXPECTED_TEST_SAMPLES,
    'test_normalized_groups': TEST_GROUP_COUNT,
    'bootstrap_group_manifest_sha256': calculate_sha256(
        TABLES_DIR / 'test_bootstrap_groups.csv'
    ),
    'thresholds_frozen': True,
    'model_parameters_frozen': True,
    'post_freeze_training_permitted': False
}

setup_path = (
    RUN_CONFIGS_DIR / 'evaluation_setup.json'
)

if setup_path.exists():

    assert read_json(setup_path) == evaluation_setup

else:

    save_json_atomic(setup_path, evaluation_setup)


# Confirm that the frozen protocol remains unchanged

assert calculate_sha256(FROZEN_MANIFEST_PATH) == (
    EXPECTED_FROZEN_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    frozen_manifest['registry']['sha256']
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    frozen_manifest['decision_log_sha256']
)

gc.collect()

print("\nFinal evaluation readiness summary")
print("Frozen models: 19")
print("Classical models: 12")
print("Transformer models: 7")
print("Official test comments: 63978")
print("Primary model:", PRIMARY_RUN_ID)
print("Test dataset and IDs: Verified")
print("Thresholds: Unchanged")
print("Model training: None")
print("Test inference performed: No")
print("GPU available:", CUDA_AVAILABLE)
print("Frozen manifest: Unchanged")

print("\nCell 1 completed successfully.")

In [ ]:
script_path = SRC_DIR / '07_evaluation_cell2.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '07_evaluation_cell3.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '07_evaluation_cell4.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '07_evaluation_cell5.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '07_evaluation_cell6.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '07_evaluation_cell7.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '07_evaluation_cell8.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())
